# Плотность токенизатора T-lite и Qwen3

T-lite-it-2.1 построена на Qwen3-8B (та же архитектура и тот же объём вычислений на токен), но с другим токенизатором:
по проверке на одной фразе русский текст у неё на треть короче, английский — такой же. Здесь считаем то же на выборке
из пилотного корпуса Википедии: 300 разделов русской и 300 разделов английской Википедии
(`data/sources/tokenizer_sample.jsonl`, разделы 500–4000 символов, по одному из статьи, seed 42).

Запуск: локально, на сервере или на Kaggle (загрузить `tokenizer_sample.jsonl` как датасет). Нужен только
`transformers>=4.51` (токенизаторы), веса моделей и PyTorch не нужны. Вывод ниже — запуск на сервере 16.10.2026.

Результат: на русском T-lite тратит 0,66 токена на каждый токен Qwen3 (медиана по разделам; 10–90%: 0,62–0,72),
то есть Qwen3 нужно в 1,5 раза больше токенов; английский текст токенизируется одинаково (отношение 1,000).
Размер словаря тот же (151 668 и 151 669).

In [2]:
import glob, json, statistics
from pathlib import Path

from transformers import AutoTokenizer

local = Path("../data/sources/tokenizer_sample.jsonl")
SAMPLE = local if local.exists() else Path(glob.glob("/kaggle/input/**/tokenizer_sample.jsonl", recursive=True)[0])
sections = [json.loads(line) for line in SAMPLE.open(encoding="utf-8")]
MODELS = {"T-lite-it-2.1": "t-tech/T-lite-it-2.1", "Qwen3-8B": "Qwen/Qwen3-8B"}
tokenizers = {name: AutoTokenizer.from_pretrained(repo) for name, repo in MODELS.items()}
print(len(sections), "разделов;", {name: len(tok) for name, tok in tokenizers.items()}, "— размер словаря")

600 разделов; {'T-lite-it-2.1': 151668, 'Qwen3-8B': 151669} — размер словаря


In [3]:
# Токены на каждый раздел обоими токенизаторами.
counts = {name: [len(tok(s["text"], add_special_tokens=False)["input_ids"]) for s in sections]
          for name, tok in tokenizers.items()}

print(f"{'язык':5} {'модель':15} {'токенов':>9} {'симв./токен':>12} {'токенов/слово':>14}")
for lang in ("ru", "en"):
    idx = [i for i, s in enumerate(sections) if s["lang"] == lang]
    chars = sum(len(sections[i]["text"]) for i in idx)
    words = sum(len(sections[i]["text"].split()) for i in idx)
    for name in MODELS:
        tokens = sum(counts[name][i] for i in idx)
        print(f"{lang:5} {name:15} {tokens:9d} {chars / tokens:12.2f} {tokens / words:14.2f}")
    ratios = [counts["T-lite-it-2.1"][i] / counts["Qwen3-8B"][i] for i in idx]
    print(f"      T-lite / Qwen3 по разделам: медиана {statistics.median(ratios):.3f}, "
          f"10–90%: {statistics.quantiles(ratios, n=10)[0]:.3f}–{statistics.quantiles(ratios, n=10)[-1]:.3f}\n")

язык  модель            токенов  симв./токен  токенов/слово
ru    T-lite-it-2.1      108170         4.30           1.77
ru    Qwen3-8B           163060         2.86           2.67
      T-lite / Qwen3 по разделам: медиана 0.660, 10–90%: 0.616–0.715

en    T-lite-it-2.1       92401         4.74           1.34
en    Qwen3-8B            92407         4.74           1.34
      T-lite / Qwen3 по разделам: медиана 1.000, 10–90%: 1.000–1.000



In [4]:
# Проверочная фраза (16.10.2026: T-lite 22 / 22, Qwen3 33 / 22).
ru = "Градиентный спуск — метод нахождения локального минимума функции с помощью движения вдоль антиградиента."
en = "Gradient descent is a method for finding a local minimum of a function by moving along the antigradient."
for name, tok in tokenizers.items():
    print(name, len(tok(ru)["input_ids"]), len(tok(en)["input_ids"]))

T-lite-it-2.1 22 22
Qwen3-8B 33 22
